In [4]:
import sys
import os

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pandas as pd


# Fetch and prepare data for machine learning

* select date or date range (less than 7 days) for prediction
* fetch energy data from SMARD for the selected time frame - 1
* fetch weather data from open-meteo of top five selected German cities for the selected time frame - 1
* calculte weather data, weighted on the population of the cities
* merge energy and weather data on 'time' column
* add features to the data set 
    - time features - date components and holiday, weekend
    - lag and rolling features of the weather data - 24 hours and 7 days 

In [24]:
# feature data for the next 24 hours
from fetch_prepare_data import fetch_smard_netzlast, fetch_weather_data_for_cities, merge_weather_data_with_city_weights, prepare_weather_data_for_modeling, combine_energy_weather_dataset, create_weather_features, create_time_based_features


prediction_date = "2026-05-07"
# Need at least 15 days of history so that the 168h lag/rolling features have enough rows
start_date = (pd.to_datetime(prediction_date) - pd.Timedelta(days=15)).strftime("%Y-%m-%d")
end_date = (pd.to_datetime(prediction_date) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

df_energy = fetch_smard_netzlast(start_date, end_date)
#print("Energy data for Germany:")
#display(df_energy.head())

weather_data_cities = fetch_weather_data_for_cities(in_start_date= start_date, in_end_date = end_date) 
df_weather_germany = merge_weather_data_with_city_weights(weather_data_cities)
df_weather_germany = prepare_weather_data_for_modeling(in_start_date = start_date, in_end_date = end_date)
#print("Weather data for Germany:")
#display(df_weather_germany.head())

df_energy_weather = combine_energy_weather_dataset(df_energy, df_weather_germany)
df_energy_weather = create_time_based_features(df_energy_weather, in_year=pd.to_datetime(prediction_date).year)
df_energy_weather = create_weather_features(df_energy_weather)
print(f"Combined energy and weather data: {df_energy_weather.shape}")
display(df_energy_weather.head())

#best_rf_model = pickle.load(open("../models/best_rf_model.pkl", "rb"))
#features = df_energy_weather.drop(columns=['load'])
#prediction = best_rf_model.predict(features)
#print(f"Predicted electricity load for {prediction_date}:")
#print(prediction)


Combined energy and weather data: (192, 24)


,DateUTC,EnergyDemand,apparent_temperature,rain,snowfall,wind_speed_10m,shortwave_radiation,weathercode,hour,day_of_week,...,apparent_temperature_lag_24h,shortwave_radiation_0m_rolling_mean_24h,shortwave_radiation_0m_lag_24h,heating_degree,cooling_degree,is_pandemic_time,EnergyDemand_lag_24h,EnergyDemand_lag_168h,EnergyDemand_rolling_mean_24h,EnergyDemand_rolling_mean_168h
168,2026-04-29 02:00:00+02:00,43516.31,5.486004,0.0,0.0,12.425247,0.0,0.501865,2,2,...,5.920169,276.649512,0.0,9.513996,0,0,42481.34,43801.04,52775.934583,50185.701845
169,2026-04-29 03:00:00+02:00,43879.10,4.860583,0.0,0.0,11.534044,0.0,0.501865,3,2,...,5.258011,276.649512,0.0,10.139417,0,0,42555.71,44313.88,52819.058333,50184.007024
170,2026-04-29 04:00:00+02:00,44844.75,4.027936,0.0,0.0,11.290396,0.0,0.501865,4,2,...,4.482496,276.649512,0.0,10.972064,0,0,44022.33,45234.16,52874.199583,50181.419048
171,2026-04-29 05:00:00+02:00,47955.23,3.278176,0.0,0.0,10.695088,0.0,0.501865,5,2,...,3.770240,276.649512,0.0,11.721824,0,0,47939.26,49140.84,52908.467083,50179.101131
172,2026-04-29 06:00:00+02:00,54863.64,2.723710,0.0,0.0,10.354328,0.0,0.501865,6,2,...,3.189719,276.649512,0.0,12.276290,0,0,54724.66,56090.78,52909.132500,50172.043929


In [25]:
df_energy_weather.to_csv(f"../data/processed/energy_weather_data_{prediction_date}.csv", index=False)

In [ ]:
# Load model and predict
# Note: best_rf_model was trained with EnergyDemand_lag_8760h and EnergyDemand_rolling_mean_8760h.
# Since those features were dropped, the model needs to be retrained before this will work.
best_rf_model = pickle.load(open("../models/best_rf_model.pkl", "rb"))

model_features = list(best_rf_model.feature_names_in_)
print("Features expected by model:", model_features)
print("Features available:", [c for c in df_energy_weather.columns if c not in ['DateUTC', 'EnergyDemand']])

# Drop non-feature columns
cols_to_drop = ['DateUTC', 'EnergyDemand', 'weathercode']
cols_to_drop = [c for c in cols_to_drop if c in df_energy_weather.columns]
features = df_energy_weather.drop(columns=cols_to_drop)

# Check for missing features
missing = [f for f in model_features if f not in features.columns]
extra = [c for c in features.columns if c not in model_features]
if missing:
    print(f"\n⚠ Missing features (model needs retraining): {missing}")
if extra:
    print(f"\n⚠ Extra columns (will be dropped): {extra}")
    features = features[model_features]


# Add interactive notebook for prediction

* country selection - dropdown
* time selection - calendar and time 


In [ ]:
from ipywidgets import interact, widgets, Output
from IPython.display import display, clear_output